# Analyzing models

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you can:

1. Run flux variability analysis (FVA) and explain what a flux range means.
2. Recognize thermodynamically infeasible loops in FVA results and remove them with loopless FVA.
3. Use `fraction_of_optimum` to explore optimal and sub-optimal solution spaces.
4. Compute and interpret a phenotypic phase plane (production envelope) for growth versus product formation.

**Estimated time:** 60 min (30 min guided, 30 min exercises).

**Prerequisites:** notebooks 01 and 02 (loading a model, running FBA, bounds and objectives).

### Background

**Why FVA?** Flux balance analysis (FBA) returns *one* flux distribution that maximizes the objective. Metabolic networks are full of redundancy (alternative pathways, isozymes, reversible reactions), so often many flux distributions reach the same optimum and FBA reports one of them more or less arbitrarily. Flux variability analysis ([Mahadevan & Schilling, 2003](https://doi.org/10.1016/j.ymben.2003.09.002)) asks, for every reaction $j$, how far its flux can move while the model still satisfies all constraints. It solves two linear programs (LPs) per reaction:

$$
v_j^{min} = \min v_j, \qquad v_j^{max} = \max v_j \qquad \text{subject to} \quad S v = 0,\; lb \le v \le ub,\; v_{obj} \ge f \cdot v_{obj}^{opt}
$$

where $f$ is the `fraction_of_optimum`: $f = 0$ means any growth rate is allowed, $f = 1$ means only optimal growth. A narrow range means the flux is tightly determined; a wide range means the network can reroute flux around that reaction.

**Loops.** Some sets of reactions can form a cycle that carries flux with no net conversion, for example one reaction running forwards while another runs the same chemistry backwards. Mass balance ($Sv = 0$) cannot rule this out, but thermodynamics does: a closed cycle has no driving force. Such thermodynamically infeasible loops show up in FVA as reactions that reach their artificial bounds (here 1000). *Loopless* FVA removes these cycles ([Schellenberger et al., 2011](https://doi.org/10.1016/j.bpj.2010.12.3707)); cobrapy offers the CycleFreeFlux method for this.

**Phenotypic phase planes.** A phenotypic phase plane (also called a production envelope) scans one flux, e.g. the growth rate, and at each point computes the minimum and maximum of another flux, e.g. acetate secretion. The enclosed area is everything the network can do. It is the standard picture for growth versus production trade-offs ([Edwards et al., 2001](https://doi.org/10.1038/84379)), and it returns in the strain design notebooks (09, 10).

In [ ]:
import escher
from cobra.io import read_sbml_model
from cobra.flux_analysis import flux_variability_analysis
model = read_sbml_model('data/e_coli_core.xml.gz')

In [ ]:
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

## Flux variability analysis (Part 1)

Calculate the flux ranges of all reactions in the _E. coli_ core model. We set `fraction_of_optimum=0`, so the growth rate may take any value between zero and its maximum. (The cobrapy default is `1.0`, i.e. optimal growth only; we come back to that in the next section.)

In [ ]:
result = flux_variability_analysis(model, fraction_of_optimum=0)

Inspect the result: one row per reaction, with the smallest (`minimum`) and largest (`maximum`) flux the reaction can carry. Negative values mean the reaction runs in reverse.

In [ ]:
result

Get an overview of a few key statistics of the resulting flux ranges. Note the maximum of 1000: that is the default bound, not a biologically meaningful flux.

In [ ]:
result.describe()

Visualize the width of each flux range (`maximum - minimum`) on a pathway map of _E. coli_ central carbon metabolism. Reactions with a very large range show up in red.

In [ ]:
abs_flux_ranges = abs(result.maximum - result.minimum).to_dict()
escher.Builder('e_coli_core.Core metabolism', reaction_data=abs_flux_ranges)

The reactions showing up in red are part of thermodynamically infeasible loops. Listing all reactions that can exceed 500 reveals two pairs:

* `FRD7` (fumarate reductase) and `SUCDi` (succinate dehydrogenase) interconvert succinate and fumarate using the quinone pool. Running both at the same time converts nothing, yet each flux can reach the bound of 1000.
* `FORt2` and `FORti` import and export formate; together they form a transport cycle.

In [ ]:
result[result.maximum > 500]

Loopless FVA removes these cycles. Compare the map below with the one above: the loop reactions no longer light up.

In [ ]:
result_no_cyles = flux_variability_analysis(model, loopless='cycleFreeFlux')

In [ ]:
abs_flux_ranges = abs(result_no_cyles.maximum - result_no_cyles.minimum).to_dict()
escher.Builder('e_coli_core.Core metabolism', reaction_data=abs_flux_ranges)

## Run flux variability analysis for optimally growing _E. coli_

FBA solutions are not necessarily unique. FVA at the optimal growth rate tells us how much freedom is left among the *alternative optimal* solutions. First, we determine the maximal growth rate and force the biomass reaction to stay at that value.

In [ ]:
fba_solution = model.optimize()

In [ ]:
fba_solution.objective_value

In [ ]:
model_optimal = model.copy()

Copying the model keeps the original unconstrained model intact. (Notebook 05 shows a faster alternative with `with model:`.)

In [ ]:
model_optimal.reactions.BIOMASS_Ecoli_core_w_GAM.lower_bound = fba_solution.objective_value

In [ ]:
result_max_obj = flux_variability_analysis(model_optimal, loopless='cycleFreeFlux')

In [ ]:
result_max_obj

This is such a common task that `flux_variability_analysis` provides the `fraction_of_optimum` option: the objective is constrained to at least that fraction of its optimum.

In [ ]:
result_max_obj = flux_variability_analysis(model, fraction_of_optimum=1., loopless='cycleFreeFlux')

In [ ]:
result_max_obj

Adding up the widths of all flux ranges gives a single number for the size of the optimal solution space. It is (numerically) zero: once loops are removed, the optimal solution of this small core model is unique. For genome-scale models this is usually not the case.

In [ ]:
sum(abs(result_max_obj.minimum - result_max_obj.maximum))

## Exercises (20 min)

### Exercise 1

Explore how relaxing the constraint on the growth rate affects the solution space:
1. Modify the code to explore flux ranges for $\mu \gt 0.7 \ h^{-1}$ 
1. Plot the sum of flux ranges over a range of percentages.

In [ ]:
# Your code here


### Exercise 2

Using FVA, determine all blocked reactions ($v = 0$) in the model.

In [ ]:
# Your code here


## Phenotypic phase plane analysis (Part 2)

Load a few packages. Cobrapy implements the same calculation as `cobra.flux_analysis.production_envelope`, but the cameo version comes with plotting built in. `cameo_compat` patches the (no longer maintained) cameo package so it runs on current Python and cobrapy.

In [ ]:
import pandas
pandas.options.display.max_rows = 12
import cameo_compat  # noqa: F401 (patches cameo for modern Python/cobra)
from cameo.flux_analysis import phenotypic_phase_plane
from cameo.visualization.plotting.with_plotly import PlotlyPlotter
plotter = PlotlyPlotter()

Compute the phenotypic phase plane for growth and acetate secretion: for a range of growth rates, what are the minimal and maximal acetate secretion fluxes?

In [ ]:
result = phenotypic_phase_plane(model,
                                variables=[model.reactions.BIOMASS_Ecoli_core_w_GAM],
                                objective=model.reactions.EX_ac_e)

Look at the result in a tabular view.

In [ ]:
result.data_frame

Plot the phenotypic phase plane showing the flux through the objective (acetate secretion). At zero growth all the glucose can go to acetate; at maximal growth no acetate *has* to be made. The lower edge tells you whether production is coupled to growth: here it is zero everywhere, so the cell can grow without making acetate.

In [ ]:
result.plot(plotter)

We may also be interested in the carbon yield: the fraction of the carbon taken up as glucose that ends up in acetate, at different growth rates.

In [ ]:
result.plot(plotter, estimate='c_yield')

We can also calculate a three dimensional phenotypic phase plane: the maximal growth rate as a function of both acetate secretion and oxygen uptake. This shows how the aerobic (high oxygen uptake) and anaerobic (low oxygen uptake) regimes differ.

In [ ]:
result_3D = phenotypic_phase_plane(model, variables=[model.reactions.EX_ac_e, model.reactions.EX_o2_e],
                                   objective=model.reactions.BIOMASS_Ecoli_core_w_GAM, points=50)

In [ ]:
result_3D.data_frame

In [ ]:
# cameo's 3D plot is incompatible with current plotly, so we draw the surface directly
import plotly.graph_objects as go

growth = result_3D.data_frame.pivot(index="EX_o2_e", columns="EX_ac_e", values="objective_upper_bound")
fig = go.Figure(go.Surface(x=growth.columns, y=growth.index, z=growth.values))
fig.update_layout(scene=dict(xaxis_title="EX_ac_e", yaxis_title="EX_o2_e", zaxis_title="max. growth rate [h⁻¹]"))
fig.show()

## Exercise (10 min)

* Use `phenotypic_phase_plane` to determine the optimal O2 uptake rate.

In [ ]:
# Your code here


## References

- Edwards, Ibarra, Palsson (2001). In silico predictions of Escherichia coli metabolic capabilities are consistent with experimental data. *Nature Biotechnology* 19:125–130. [doi.org/10.1038/84379](https://doi.org/10.1038/84379)
- Mahadevan, Schilling (2003). The effects of alternate optimal solutions in constraint-based genome-scale metabolic models. *Metabolic Engineering* 5:264–276. [doi.org/10.1016/j.ymben.2003.09.002](https://doi.org/10.1016/j.ymben.2003.09.002)
- Schellenberger, Lewis, Palsson (2011). Elimination of Thermodynamically Infeasible Loops in Steady-State Metabolic Models. *Biophysical Journal* 100:544–553. [doi.org/10.1016/j.bpj.2010.12.3707](https://doi.org/10.1016/j.bpj.2010.12.3707)